<a href="https://colab.research.google.com/github/fhmahi957/Licence-plate-NightVision/blob/main/Licence_plate_NightVision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1
from google.colab import drive
import os
import torch

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Create a folder for your models
save_dir = '/content/drive/MyDrive/CSE414_Project2/models'
os.makedirs(save_dir, exist_ok=True)

# 3. Save the trained model weights to Drive
model_path = os.path.join(save_dir, 'faster_rcnn_baseline.pth')
torch.save(model.state_dict(), model_path)

print(f"SUCCESS! Model saved to Google Drive at: {model_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
SUCCESS! Model saved to Google Drive at: /content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline.pth


In [9]:
# Cell 2

# Load cell

# Last run

# 1. Mount Drive and Import Libraries
from google.colab import drive
import torch
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

drive.mount('/content/drive')

# 2. Recreate the Model Architecture (The "Empty Kitchen")
# We must define the exact same architecture we trained
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights=None, weights_backbone=None,min_size=640, max_size=640)
in_features = model.roi_heads.box_predictor.cls_score.in_features
# NUM_CLASSES was 4 (Background + 3 objects from your dataset)
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 4)

# 3. Load the Weights from Drive (Bringing the "Cake" out of the fridge)
model_path = '/content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline_v3.pth'
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

# 4. Set to Evaluation Mode
model.eval()
print("SUCCESS! Model loaded from Google Drive and ready for testing.")

Mounted at /content/drive
SUCCESS! Model loaded from Google Drive and ready for testing.


In [ ]:
# Cell 3
# Evaluation code

# import os
# import torch
# from torch.utils.data import DataLoader, Dataset
# from PIL import Image
# import torchvision.transforms as T
# from torchmetrics.detection import MeanAveragePrecision
# from tqdm import tqdm

# # Paths (to fix data leakage)
# NIGHT_IMG_DIR = '/content/night_test_images'
# LBL_DIR = '/content/test_labels' # Using test labels, not source labels

# # 1. Define the Nighttime Dataset
# # We iterate through the ORIGINAL filenames, but load the NIGHTTIME images
# class NighttimeDataset(Dataset):
#     def __init__(self, night_img_dir, lbl_dir):
#         self.night_img_dir = night_img_dir
#         self.lbl_dir = lbl_dir
#         # Get list of original filenames (e.g., 'car1.jpg')
#         # We assume the night images are named 'night_car1.jpg'
#         self.original_images = sorted([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])

#     def __len__(self):
#         return len(self.original_images)

#     def __getitem__(self, idx):
#         orig_name = self.original_images[idx] # e.g., 'car1.txt'
#         img_name = orig_name.replace('.txt', '.jpg') # e.g., 'car1.jpg'
#         night_img_name = f"night_{img_name}" # e.g., 'night_car1.jpg'

#         img_path = os.path.join(self.night_img_dir, night_img_name)
#         lbl_path = os.path.join(self.lbl_dir, orig_name)

#         image = Image.open(img_path).convert("RGB")
#         width, height = image.size

#         boxes = []
#         labels = []

#         if os.path.exists(lbl_path):
#             with open(lbl_path, 'r') as f:
#                 for line in f.readlines():
#                     parts = line.strip().split()
#                     if len(parts) == 5:
#                         class_id = int(parts[0])
#                         x_center = float(parts[1])
#                         y_center = float(parts[2])
#                         w = float(parts[3])
#                         h = float(parts[4])

#                         x1 = (x_center - w / 2) * width
#                         y1 = (y_center - h / 2) * height
#                         x2 = (x_center + w / 2) * width
#                         y2 = (y_center + h / 2) * height

#                         # Clamp
#                         x1 = max(0, x1); y1 = max(0, y1)
#                         x2 = min(width, x2); y2 = min(height, y2)

#                         if x2 > x1 and y2 > y1:
#                             boxes.append([x1, y1, x2, y2])
#                             labels.append(class_id + 1) # 1-indexed

#         if len(boxes) == 0:
#             boxes = torch.zeros((0, 4), dtype=torch.float32)
#             labels = torch.zeros((0,), dtype=torch.int64)
#         else:
#             boxes = torch.as_tensor(boxes, dtype=torch.float32)
#             labels = torch.as_tensor(labels, dtype=torch.int64)

#         target = {
#             'boxes': boxes,
#             'labels': labels,
#             'image_id': torch.tensor([idx]),
#             'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
#             'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
#         }

#         # Transform to tensor (same as training)
#         transform = T.Compose([T.ToTensor()])
#         image = transform(image)

#         return image, target

# # 2. Setup Data Loader
# dataset = NighttimeDataset(NIGHT_IMG_DIR, LBL_DIR)
# # Batch size 1 is best for evaluation metrics
# data_loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))

# device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model.to(device)
# model.eval() # Crucial for evaluation

# # 3. Initialize Metric
# metric = MeanAveragePrecision(iou_type="bbox", box_format="xyxy")

# print("Starting Domain Shift Analysis (Testing Clean Model on Nighttime Data)...")
# print("This measures the 'Before' performance.\n")

# # 4. Evaluation Loop
# with torch.no_grad():
#     for images, targets in tqdm(data_loader, desc="Evaluating"):
#         images = list(image.to(device) for image in images)
#         targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

#         # Get predictions
#         predictions = model(images)

#         # Format for torchmetrics
#         # predictions is a list of dicts: [{'boxes':..., 'labels':..., 'scores':...}]
#         preds_for_metric = [{
#             'boxes': p['boxes'].cpu(),
#             'labels': p['labels'].cpu(),
#             'scores': p['scores'].cpu()
#         } for p in predictions]

#         targets_for_metric = [{
#             'boxes': t['boxes'].cpu(),
#             'labels': t['labels'].cpu()
#         } for t in targets]

#         # Update metric
#         metric.update(preds_for_metric, targets_for_metric)

# # 5. Compute and Print Results
# result = metric.compute()
# print("\n DOMAIN SHIFT ANALYSIS RESULTS")
# print(f"mAP (mean Average Precision): {result['map'].item():.4f}")
# print(f"mAP @ 0.5 IoU: {result['map_50'].item():.4f}")
# print(f"mAP @ 0.75 IoU: {result['map_75'].item():.4f}")

# print("Note: A low mAP here proves that Domain Shift is a problem!")


Starting Domain Shift Analysis (Testing Clean Model on Nighttime Data)...
This measures the 'Before' performance.



Evaluating: 100%|██████████| 99/99 [00:05<00:00, 18.47it/s]



 DOMAIN SHIFT ANALYSIS RESULTS
mAP (mean Average Precision): 0.2154
mAP @ 0.5 IoU: 0.3234
mAP @ 0.75 IoU: 0.2376
Note: A low mAP here proves that Domain Shift is a problem!


In [ ]:
# Cell mr bean

# 1st run

# Installing necessary libraries
!pip install ultralytics albumentations opencv-python-headless matplotlib seaborn tqdm pyyaml torchmetrics -q

In [ ]:
# Cell 4
import os
import cv2
import torch
import albumentations as A
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Check if the GPU is connected and ready
print("Environment setup complete!")
print(f"PyTorch version: {torch.__version__}")
print(f"Is GPU available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not available. Please go to Runtime > Change runtime type and select T4 GPU.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Environment setup complete!
PyTorch version: 2.11.0+cu130
Is GPU available: True
GPU Name: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os


print("Contents of MyDrive:")
print(os.listdir('/content/drive/MyDrive/'))


project_path = '/content/drive/MyDrive/CSE414_Project2'
if os.path.exists(project_path):
    print(f"\nFound project folder! Contents:")
    print(os.listdir(project_path))
else:
    print("\nCSE414_Project2 folder not found in MyDrive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Contents of MyDrive:
['wallpaperflare.com_wallpaper.jpg', 'Yearbook.gslides', 'Untitled presentation (1).gslides', 'Untitled presentation.gslides', 'Google AI Studio', 'Untitled document (2).gdoc', '23101182_report2 .gdoc', 'Untitled document (1).gdoc', '5.gdoc', '7.gdoc', 'Queries for B2B automation platform.gdoc', '23101182,23101183,23101186,23101189.docx', 'Overview of Smart Water Tank project .gdoc', ' 23101186_Crypto-Algo-Imply.gdoc', '23101182_Crypto-Algo-Imply.gdoc', 'Lead Capture (updated code files).gdoc', 'Dashboard & State api (updated code).gdoc', 'Auth and rbac(updated code).gdoc', 'API-Documentation (Group-3).gdoc', 'Apology mail for delay .gsheet', 'Scheduled_Emails.gsheet', 'Untitled document.gdoc', 'Classroom', 'Tasnim Rahman Tayeba.gslides', 'Fire Fighting Robot (Group-5).gslides', 'Colab Notebooks', 'CSE414_Project2']

Found project folder!

In [ ]:
# Cell 5

from google.colab import files
import zipfile
import os
import shutil

# 1. Upload zip file from pc
print("Please upload the Roboflow ZIP file from your computer...")
uploaded = files.upload()
zip_filename = list(uploaded.keys())[0]
print(f"Uploaded: {zip_filename}")


for folder in ['/content/train_images', '/content/train_labels',
               '/content/test_images', '/content/test_labels',
               '/content/dataset_raw']:
    shutil.rmtree(folder, ignore_errors=True)


os.makedirs('/content/train_images', exist_ok=True)
os.makedirs('/content/train_labels', exist_ok=True)
os.makedirs('/content/test_images', exist_ok=True)
os.makedirs('/content/test_labels', exist_ok=True)

# 4. Unzip and organize
print("\nUnzipping and splitting data...")
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall('/content/dataset_raw')

base_path = '/content/dataset_raw'

for root, dirs, files in os.walk(base_path):
    # Train Split
    if 'images' in dirs and 'train' in root:
        for img in os.listdir(os.path.join(root, 'images')):
            if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.move(os.path.join(root, 'images', img), '/content/train_images/')
    if 'labels' in dirs and 'train' in root:
        for lbl in os.listdir(os.path.join(root, 'labels')):
            if lbl.lower().endswith('.txt'):
                shutil.move(os.path.join(root, 'labels', lbl), '/content/train_labels/')

    # Test Split
    if 'images' in dirs and 'test' in root:
        for img in os.listdir(os.path.join(root, 'images')):
            if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.move(os.path.join(root, 'images', img), '/content/test_images/')
    if 'labels' in dirs and 'test' in root:
        for lbl in os.listdir(os.path.join(root, 'labels')):
            if lbl.lower().endswith('.txt'):
                shutil.move(os.path.join(root, 'labels', lbl), '/content/test_labels/')

print(f"\nData Split Done!")
print(f"Train Images: {len(os.listdir('/content/train_images'))}")
print(f"Train Labels: {len(os.listdir('/content/train_labels'))}")
print(f"Test Images: {len(os.listdir('/content/test_images'))}")
print(f"Test Labels: {len(os.listdir('/content/test_labels'))}")

Please upload the Roboflow ZIP file from your computer...


Saving bangla-license-plate-detection.v1i.yolov8.zip to bangla-license-plate-detection.v1i.yolov8.zip
Uploaded: bangla-license-plate-detection.v1i.yolov8.zip

Unzipping and splitting data...

Data Split Done!
Train Images: 746
Train Labels: 746
Test Images: 99
Test Labels: 99


In [5]:
#Cell 6

# run 3rd

# taking new images form test folder
# creating distorted image from clean image
import os, cv2, random, numpy as np, albumentations as A
from tqdm import tqdm

random.seed(42); np.random.seed(42)

CLEAN_TEST_DIR = '/content/test_images'
NIGHT_TEST_DIR = '/content/night_test_images'
os.makedirs(NIGHT_TEST_DIR, exist_ok=True)

nighttime_transform = A.Compose([
    A.RandomGamma(gamma_limit=(150, 250), p=0.9),
    A.RandomBrightnessContrast(brightness_limit=(-0.5, -0.2), contrast_limit=(-0.4, 0.0), p=0.9),
    A.GaussianBlur(blur_limit=(3, 7), p=0.6),
    A.MotionBlur(blur_limit=(5, 11), p=0.5),
])

def make_night(image_rgb):
    out = nighttime_transform(image=image_rgb)['image'].astype(np.float32)
    out += np.random.normal(0, np.random.uniform(5, 20), out.shape)
    return np.clip(out, 0, 255).astype(np.uint8)

image_files = [f for f in os.listdir(CLEAN_TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

for f in tqdm(image_files):
    img = cv2.cvtColor(cv2.imread(os.path.join(CLEAN_TEST_DIR, f)), cv2.COLOR_BGR2RGB)
    cv2.imwrite(os.path.join(NIGHT_TEST_DIR, f"night_{f}"),
                cv2.cvtColor(make_night(img), cv2.COLOR_RGB2BGR))

print(f"\nSuccess! Generated {len(image_files)} nighttime images in {NIGHT_TEST_DIR}")

100%|██████████| 99/99 [00:04<00:00, 22.66it/s]


Success! Generated 99 nighttime images in /content/night_test_images


In [ ]:
#Cell 7

#checking class ID's
import os

LBL_DIR = '/content/train_labels'
class_ids = set()

for lbl_file in os.listdir(LBL_DIR):
    if lbl_file.endswith('.txt'):
        with open(os.path.join(LBL_DIR, lbl_file), 'r') as f:
            for line in f.readlines():
                parts = line.strip().split()
                if len(parts) >= 1:
                    class_ids.add(int(parts[0]))

print(f"Found class IDs in dataset: {sorted(class_ids)}")
print(f"Max class ID: {max(class_ids) if class_ids else 0}")

Found class IDs in dataset: [0, 1, 2]
Max class ID: 2


In [ ]:
# Cell 8


import os
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_320_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import cv2
import numpy as np
import albumentations as A

IMG_DIR = '/content/train_images'
LBL_DIR = '/content/train_labels'


class LicensePlateDataset(Dataset):
    def __init__(self, img_dir, lbl_dir, augment=False):
        self.img_dir = img_dir
        self.lbl_dir = lbl_dir
        self.images = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        self.augment = augment

        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.Resize(640, 640),
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))
        else:
            self.transform = A.Compose([
                A.Resize(640, 640),
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        lbl_path = os.path.join(self.lbl_dir, img_name.rsplit('.', 1)[0] + '.txt')

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        boxes = []
        labels = []

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                for line in f.readlines():
                    parts = line.strip().split()
                    if len(parts) == 5:
                        class_id = int(parts[0])
                        x_center = float(parts[1])
                        y_center = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])

                        x1 = (x_center - w / 2) * width
                        y1 = (y_center - h / 2) * height
                        x2 = (x_center + w / 2) * width
                        y2 = (y_center + h / 2) * height

                        x1 = max(0, x1); y1 = max(0, y1)
                        x2 = min(width, x2); y2 = min(height, y2)

                        if x2 > x1 and y2 > y1:
                            boxes.append([x1, y1, x2, y2])
                            labels.append(class_id + 1)

        transformed = self.transform(image=image, bboxes=boxes, labels=labels)
        image_np = transformed['image']
        boxes = transformed['bboxes']
        labels = transformed['labels']

        image = torchvision.transforms.ToTensor()(image_np)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
        }

        return image, target


# 2. Setup Data Loader
dataset = LicensePlateDataset(IMG_DIR, LBL_DIR, augment=True)
data_loader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=lambda x: tuple(zip(*x)))

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"Using device: {device}")

print("Loading model...")
model = fasterrcnn_mobilenet_v3_large_320_fpn(weights="DEFAULT", min_size=640, max_size=640)
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 4)
model.to(device)

# 3. Hyperparameters
params = [p for p in model.parameters() if p.requires_grad]
optimizer = optim.SGD(params, lr=0.003, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

num_epochs = 100
print(f"Starting training for {num_epochs} epochs...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for images, targets in data_loader:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        epoch_loss += losses.item()

    lr_scheduler.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs} | Loss: {epoch_loss/len(data_loader):.4f}")

print("\nTraining complete!")

torch.save(model.state_dict(), '/content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline_v3.pth')
print("SUCCESS! v3 model saved to Google Drive.")

Using device: cuda
Loading model...
Starting training for 100 epochs...
Epoch 10/100 | Loss: 0.3406
Epoch 20/100 | Loss: 0.3338
Epoch 30/100 | Loss: 0.3146
Epoch 40/100 | Loss: 0.2642
Epoch 50/100 | Loss: 0.2013
Epoch 60/100 | Loss: 0.1631
Epoch 70/100 | Loss: 0.1240
Epoch 80/100 | Loss: 0.0976
Epoch 90/100 | Loss: 0.0916
Epoch 100/100 | Loss: 0.0885

Training complete!
SUCCESS! v3 model saved to Google Drive.


In [ ]:
# Save the newly trained model to Google Drive
torch.save(model.state_dict(), '/content/drive/MyDrive/CSE414_Project2/models/faster_rcnn_baseline_v2.pth')
print("SUCCESS! New model saved to Google Drive.")

SUCCESS! New model saved to Google Drive.


In [ ]:
# Cell 9

# FSRCNN architecture

import torch
import torch.nn as nn

class FSRCNN(nn.Module):
    def __init__(self, scale_factor=1, d=56, s=12, m=4):
        super(FSRCNN, self).__init__()
        self.scale_factor = scale_factor

        # 1. Feature extraction
        self.feature_extraction = nn.Sequential(
            nn.Conv2d(3, d, kernel_size=5, padding=2),
            nn.PReLU()
        )

        # 2. Shrinking
        self.shrinking = nn.Sequential(
            nn.Conv2d(d, s, kernel_size=1),
            nn.PReLU()
        )

        # 3. Non-linear mapping
        self.mapping = []
        for _ in range(m):
            self.mapping.append(nn.Conv2d(s, s, kernel_size=3, padding=1))
            self.mapping.append(nn.PReLU())
        self.mapping = nn.Sequential(*self.mapping)

        # 4. Expanding
        self.expanding = nn.Sequential(
            nn.Conv2d(s, d, kernel_size=1),
            nn.PReLU()
        )

        # 5. Reconstruction (Deconvolution)
        # Since scale_factor=1 for enhancement, we use regular Conv2d
        self.reconstruction = nn.Conv2d(d, 3, kernel_size=9, padding=4)

        self._initialize_weights()

  def forward(self, x):
    identity = x
    x = self.feature_extraction(x)
    x = self.shrinking(x)
    x = self.mapping(x)
    x = self.expanding(x)
    x = self.reconstruction(x)
    return identity + x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.normal_(m.weight.data, mean=0.0, std=0.001)
                if m.bias is not None:
                    nn.init.constant_(m.bias.data, 0.0)

print("FSRCNN Architecture defined successfully!")

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 44)

In [ ]:
# Cell 10

import os, cv2, torch, numpy as np
import torch.nn as nn, torch.optim as optim
from torch.utils.data import Dataset, DataLoader

TRAIN_DIR = '/content/train_images'

class EnhancementDataset(Dataset):
    def __init__(self, clean_dir, crop=256, size=640):
        self.dir, self.crop, self.size = clean_dir, crop, size
        self.files = sorted(f for f in os.listdir(clean_dir)
                            if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        clean = cv2.cvtColor(cv2.imread(os.path.join(self.dir, self.files[i])), cv2.COLOR_BGR2RGB)
        night = make_night(clean)
        clean = cv2.resize(clean, (self.size, self.size))
        night = cv2.resize(night, (self.size, self.size))
        y = np.random.randint(0, self.size - self.crop + 1)
        x = np.random.randint(0, self.size - self.crop + 1)
        to_t = lambda im: torch.from_numpy(im[y:y+self.crop, x:x+self.crop].copy()).permute(2, 0, 1).float() / 255
        return to_t(night), to_t(clean)

train_loader = DataLoader(EnhancementDataset(TRAIN_DIR), batch_size=8, shuffle=True, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_fsrcnn = FSRCNN(scale_factor=1).to(device)
criterion = nn.L1Loss()
optimizer = optim.Adam(model_fsrcnn.parameters(), lr=5e-4)

epochs = 60
for epoch in range(epochs):
    model_fsrcnn.train()
    loss_sum = base_sum = 0
    for night_imgs, clean_imgs in train_loader:
        night_imgs, clean_imgs = night_imgs.to(device), clean_imgs.to(device)
        optimizer.zero_grad()
        loss = criterion(model_fsrcnn(night_imgs), clean_imgs)
        loss.backward(); optimizer.step()
        loss_sum += loss.item()
        base_sum += criterion(night_imgs, clean_imgs).item()
    if (epoch + 1) % 10 == 0:
        n = len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | FSRCNN: {loss_sum/n:.4f} | without enhancement: {base_sum/n:.4f}")

torch.save(model_fsrcnn.state_dict(), '/content/drive/MyDrive/CSE414_Project2/models/fsrcnn_enhancer_v2.pth')

Training FSRCNN for 50 epochs to learn Night -> Clean mapping...
Epoch 10/50 | Loss: 0.0526
Epoch 20/50 | Loss: 0.0513
Epoch 30/50 | Loss: 0.0494
Epoch 40/50 | Loss: 0.0469
Epoch 50/50 | Loss: 0.0485

FSRCNN Training Complete! Model saved to Drive.


In [ ]:
# Cell 11


# import os
# import torch
# from torch.utils.data import DataLoader, Dataset
# from PIL import Image
# import torchvision.transforms as T
# from torchmetrics.detection import MeanAveragePrecision
# from tqdm import tqdm
# import cv2
# import numpy as np
# import albumentations as A

# # Paths for Clean Test Data
# CLEAN_TEST_IMG_DIR = '/content/test_images'
# CLEAN_TEST_LBL_DIR = '/content/test_labels'

# # 1. Define the Clean Test Dataset WITH Albumentations (handles both image and bbox resize)
# class CleanTestDataset(Dataset):
#     def __init__(self, img_dir, lbl_dir):
#         self.img_dir = img_dir
#         self.lbl_dir = lbl_dir
#         self.images = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

#         # FIX: Use Albumentations to resize both image AND bounding boxes
#         self.transform = A.Compose([
#             A.Resize(640, 640),
#         ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

#     def __len__(self):
#         return len(self.images)

#     def __getitem__(self, idx):
#         img_name = self.images[idx]
#         img_path = os.path.join(self.img_dir, img_name)
#         lbl_path = os.path.join(self.lbl_dir, img_name.rsplit('.', 1)[0] + '.txt')

#         image = cv2.imread(img_path)
#         image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
#         height, width = image.shape[:2]

#         boxes = []
#         labels = []

#         if os.path.exists(lbl_path):
#             with open(lbl_path, 'r') as f:
#                 for line in f.readlines():
#                     parts = line.strip().split()
#                     if len(parts) == 5:
#                         class_id = int(parts[0])
#                         x_center = float(parts[1])
#                         y_center = float(parts[2])
#                         w = float(parts[3])
#                         h = float(parts[4])

#                         x1 = (x_center - w / 2) * width
#                         y1 = (y_center - h / 2) * height
#                         x2 = (x_center + w / 2) * width
#                         y2 = (y_center + h / 2) * height

#                         x1 = max(0, x1); y1 = max(0, y1)
#                         x2 = min(width, x2); y2 = min(height, y2)

#                         if x2 > x1 and y2 > y1:
#                             boxes.append([x1, y1, x2, y2])
#                             labels.append(class_id + 1)

#         if len(boxes) == 0:
#             boxes = [[0, 0, 1, 1]]
#             labels = [0]

#         # Apply Albumentations (resizes both image and bboxes)
#         transformed = self.transform(image=image, bboxes=boxes, labels=labels)
#         image = transformed['image']
#         boxes = transformed['bboxes']
#         labels = transformed['labels']

#         # Convert to tensors
#         image = T.ToTensor()(image)
#         boxes = torch.as_tensor(boxes, dtype=torch.float32)
#         labels = torch.as_tensor(labels, dtype=torch.int64)

#         target = {
#             'boxes': boxes,
#             'labels': labels,
#             'image_id': torch.tensor([idx]),
#             'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
#             'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64)
#         }

#         return image, target

# # 2. Setup Data Loader
# clean_dataset = CleanTestDataset(CLEAN_TEST_IMG_DIR, CLEAN_TEST_LBL_DIR)
# clean_data_loader = DataLoader(clean_dataset, batch_size=1, shuffle=False, num_workers=0, collate_fn=lambda x: tuple(zip(*x)))

# # Ensure model is on the correct device and in eval mode
# device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
# model.to(device)
# model.eval()

# # 3. Initialize Metric
# metric = MeanAveragePrecision(iou_type="bbox", box_format="xyxy")

# print("Starting Clean Baseline Evaluation (Testing Model on Clean Test Data)...")
# print("This measures the 'Perfect Condition' performance.\n")

# # 4. Evaluation Loop
# with torch.no_grad():
#     for images, targets in tqdm(clean_data_loader, desc="Evaluating Clean Images"):
#         images = list(image.to(device) for image in images)
#         targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

#         predictions = model(images)

#         preds_for_metric = [{
#             'boxes': p['boxes'].cpu(),
#             'labels': p['labels'].cpu(),
#             'scores': p['scores'].cpu()
#         } for p in predictions]

#         targets_for_metric = [{
#             'boxes': t['boxes'].cpu(),
#             'labels': t['labels'].cpu()
#         } for t in targets]

#         metric.update(preds_for_metric, targets_for_metric)

# # 5. Compute and Print Results
# result = metric.compute()
# print("\n--- CLEAN BASELINE RESULTS ---")
# print(f"Clean mAP (mean Average Precision): {result['map'].item():.4f}")
# print(f"Clean mAP @ 0.5 IoU: {result['map_50'].item():.4f}")
# print(f"Clean mAP @ 0.75 IoU: {result['map_75'].item():.4f}")
# print("------------------------------")

# # Calculate the Drop
# night_map = 0.2154 # Your previous result (calculated WITHOUT resize)
# clean_map = result['map'].item()
# drop = clean_map - night_map

# print(f"\nDomain Shift Analysis:")
# print(f"Clean mAP: {clean_map:.4f}")
# print(f"Night mAP: {night_map:.4f}")
# print(f"Performance Drop due to Domain Shift: {drop:.4f} (or {drop*100:.2f}%)")
# print("Note: This drop is the core problem your FSRCNN solution will fix!")

Starting Clean Baseline Evaluation (Testing Model on Clean Test Data)...
This measures the 'Perfect Condition' performance.



Evaluating Clean Images: 100%|██████████| 99/99 [00:02<00:00, 39.11it/s]



--- CLEAN BASELINE RESULTS ---
Clean mAP (mean Average Precision): 0.2860
Clean mAP @ 0.5 IoU: 0.4129
Clean mAP @ 0.75 IoU: 0.3110
------------------------------

Domain Shift Analysis:
Clean mAP: 0.2860
Night mAP: 0.2154
Performance Drop due to Domain Shift: 0.0706 (or 7.06%)
Note: This drop is the core problem your FSRCNN solution will fix!


In [10]:
import os, cv2, torch, numpy as np, albumentations as A
from torch.utils.data import Dataset, DataLoader
from torchmetrics.detection import MeanAveragePrecision
from tqdm import tqdm

CLEAN_TEST_DIR = '/content/test_images'
TEST_LBL_DIR   = '/content/test_labels'
NIGHT_TEST_DIR = '/content/night_test_images'
IMG_SIZE = 640
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class DetDataset(Dataset):
    def __init__(self, img_dir, prefix=''):
        self.img_dir, self.prefix = img_dir, prefix
        self.files = sorted(f for f in os.listdir(CLEAN_TEST_DIR)
                            if f.lower().endswith(('.jpg', '.jpeg', '.png')))
        self.tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE)],
                            bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    def __len__(self): return len(self.files)

    def __getitem__(self, i):
        name = self.files[i]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, self.prefix + name)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        boxes, labels = [], []
        lbl = os.path.join(TEST_LBL_DIR, name.rsplit('.', 1)[0] + '.txt')
        if os.path.exists(lbl):
            for line in open(lbl):
                p = line.split()
                if len(p) == 5:
                    c, xc, yc, bw, bh = int(p[0]), *map(float, p[1:])
                    x1 = max(0, (xc - bw/2) * w); y1 = max(0, (yc - bh/2) * h)
                    x2 = min(w, (xc + bw/2) * w); y2 = min(h, (yc + bh/2) * h)
                    if x2 > x1 and y2 > y1:
                        boxes.append([x1, y1, x2, y2]); labels.append(c + 1)
        out = self.tf(image=img, bboxes=boxes, labels=labels)
        image = torch.from_numpy(out['image']).permute(2, 0, 1).float() / 255

        b = torch.as_tensor(out['bboxes'], dtype=torch.float32).reshape(-1, 4)
        l = torch.as_tensor(out['labels'], dtype=torch.int64)
        return image, {'boxes': b, 'labels': l}

@torch.no_grad()
def evaluate(img_dir, prefix='', enhancer=None, name=''):
    dl = DataLoader(DetDataset(img_dir, prefix), batch_size=1, shuffle=False,
                    collate_fn=lambda x: tuple(zip(*x)))
    metric = MeanAveragePrecision(iou_type='bbox', box_format='xyxy', class_metrics=True)
    model.to(device).eval()
    if enhancer is not None: enhancer.to(device).eval()
    for images, targets in tqdm(dl, desc=name):
        images = [im.to(device) for im in images]
        if enhancer is not None:
            images = [enhancer(im.unsqueeze(0)).squeeze(0).clamp(0, 1) for im in images]
        preds = model(images)
        metric.update([{k: v.cpu() for k, v in p.items()} for p in preds],
                      [{k: v.cpu() for k, v in t.items()} for t in targets])
    r = metric.compute()
    print(f"[{name}] mAP={r['map']:.4f} | mAP@50={r['map_50']:.4f} | mAP@75={r['map_75']:.4f}")
    print(f"[{name}] mAP according to class : {r['map_per_class']}")
    return r

r_clean = evaluate(CLEAN_TEST_DIR, '', None, 'CLEAN')
r_night = evaluate(NIGHT_TEST_DIR, 'night_', None, 'NIGHT')
print(f"Drop (mAP@50): {r_clean['map_50'] - r_night['map_50']:.4f}")

CLEAN: 100%|██████████| 99/99 [00:05<00:00, 18.16it/s]


[CLEAN] mAP=0.3699 | mAP@50=0.4600 | mAP@75=0.4027
[CLEAN] mAP according to class : tensor([0.0000, 0.7398])


NIGHT: 100%|██████████| 99/99 [00:02<00:00, 39.56it/s]


[NIGHT] mAP=0.1220 | mAP@50=0.1615 | mAP@75=0.1471
[NIGHT] mAP according to class : tensor([0.0000, 0.2440])
Drop (mAP@50): 0.2985
